# 07 · Train CFFM-Net

The method itself, in image mode: two YOLO26-n backbones, a reliability-weighted sum at stride 4, CMFM blocks (reliability-gated cross-modal selective scan, offset alignment, local branch) at strides 8, 16 and 32, and the YOLO26 neck with a stride-4 detection head. Trained on LLVIP and on M3FD with exactly the settings of the baselines.

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot`, raw LLVIP and M3FD (or the output of 02) |
| **Expected runtime** | about 3 to 3.5 hours on 2 x T4 |
| **Produces** | checkpoints in `runs/<run>/`, full-set metrics in `runs/eval/<run>/metrics.json` |

In [ ]:
# --------------------------------------------------------------------------------------------
# Setup: the same cell opens every notebook. It finds the code, installs it, and checks the GPU.
#
# On Kaggle the uploaded cffm-net-pilot.zip appears, already unzipped, somewhere under
# /kaggle/input. Inputs are read-only, so we copy the project to /kaggle/working and install
# it from there ("pip install -e" makes `import cffm` work in this kernel, in DataLoader
# workers, and in the extra processes Ultralytics starts when it trains on two GPUs).
# Locally, notebooks live in cffm-net-pilot/notebooks/<phase>/; the project is the first folder up
# that contains src/cffm.
# --------------------------------------------------------------------------------------------
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    # up to five levels deep: /kaggle/input/<slug>/, /kaggle/input/<slug>/cffm-net-pilot/,
    # /kaggle/input/datasets/<owner>/<slug>/..., whichever layout this Kaggle version uses
    hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
            if (Path(p).parent / "src" / "cffm").is_dir()]
    assert hits, "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        for p in [PROJECT, *PROJECT.rglob("*")]:     # the input was read-only; our copy must be writable
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle"], check=True)   # cloudpickle: two-GPU launcher
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)   # must be inside PROJECT
env = cenv.setup()                       # prints platform, GPUs and paths; moves into the project folder
plan = pipeline.load_plan(env.project)   # configs/pilot.yaml: every run and its settings

## The runs

Straight from `configs/pilot.yaml`. Batch is the total over both GPUs.

In [ ]:
specs = pipeline.runs_for(plan, "07")

import pandas as pd
cols = ["name", "model", "data", "epochs", "imgsz", "batch"]
pd.DataFrame(specs)[cols]

## Train and evaluate

In [ ]:
# Train every run of this notebook, then evaluate each on the FULL validation set.
# If the Kaggle session dies halfway, just run the notebook again: finished runs are reused
# and an unfinished run resumes from its last checkpoint (see pipeline.train_from_spec).
results = [pipeline.train_from_spec(s, env) for s in specs]

## Learning curves

In [ ]:
# Learning curves straight from Ultralytics' results.csv. Two things to look for:
#   - validation mAP still climbing at the last epoch -> the pilot schedule is short (expected)
#   - training loss going down while validation mAP goes down too -> overfitting (not expected in 30 epochs)
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for s in specs:
    f = env.runs / s["name"] / "results.csv"
    if not f.exists():
        continue
    df = pd.read_csv(f)
    df.columns = [c.strip() for c in df.columns]
    axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], label=s["name"])
    axes[1].plot(df["epoch"], df["metrics/mAP50(B)"], label=s["name"])
    loss_cols = [c for c in df.columns if c.startswith("train/") and c.endswith("loss")]
    axes[2].plot(df["epoch"], df[loss_cols].sum(axis=1), label=s["name"])
for a, t in zip(axes, ["val mAP50-95 (quarter val set)", "val mAP50", "train loss (sum)"]):
    a.set_title(t); a.set_xlabel("epoch"); a.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Results on the full validation set

In [ ]:
# The numbers that matter, from the full validation set. AP is COCO AP50-95.
# AP_vt / AP_t / AP_s are the AI-TOD size bands (<8, 8-16, 16-32 px); mAP_small is COCO's <32 px.
import pandas as pd
keys = ["mAP50-95(B)", "mAP50(B)", "mAP_small(B)", "AP_vt(B)", "AP_t(B)", "AP_s(B)", "AP_m(B)", "AP_l(B)"]
rows = [{"run": r["run"], **{k.replace("(B)", ""): r["metrics"].get(k) for k in keys}} for r in results]
pd.DataFrame(rows).set_index("run").round(4)

## What it sees

Predictions (green) against ground truth (red) on night scenes from the validation set, then the reliability maps at stride 8: where the model trusts visible, and where thermal. In a dark scene, thermal reliability should be high on people and visible reliability should drop.

In [ ]:
import matplotlib.pyplot as plt
from cffm.train import _load
from cffm.viz import draw_pair, plot_reliability, predict_pair, reliability_maps

model = _load(env.runs / "llvip_cffm" / "weights" / "best.pt").to("cuda:0")
names = model.names
val = sorted((env.data / "llvip" / "images" / "visible" / "val").glob("*.jpg"))
for v in val[200:2000:450]:
    det, pair = predict_pair(model, str(v), conf=0.3)
    h, w = pair.shape[:2]; gt = []
    for line in (env.data / "llvip" / "labels" / "visible" / "val" / (v.stem + ".txt")).read_text().splitlines():
        c, x, y, bw, bh = map(float, line.split())
        gt.append([(x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h])
    plt.figure(figsize=(14, 4.5)); plt.imshow(draw_pair(pair, det, names, gt)); plt.axis("off"); plt.title(v.name)
    plt.show()

In [ ]:
for v in val[300:1500:600]:
    maps, lb = reliability_maps(model, str(v))
    plot_reliability(maps, lb, level=3); plt.show()

### Keep the results

On Kaggle, click **Save Version → Save & Run All (Commit)**. The notebook then reruns top to bottom in the
background, and its `runs/` folder becomes an output that notebooks **09** and **15** can attach as an input
(*Add Input → Your Work → this notebook*). Without a saved version, the checkpoints vanish when the session ends.